# 2_2_04 — Silver: lead activities envelope child table

Unpivots the `envelope` struct of `silver.lead_activites_processed` into `silver.lead_activites_processed_envelope` — one row per envelope email across all 6 directions (from/to/cc/bcc/reply_to/sender), carrying `email`, `name`, `envelope_subject`, `envelope_message_id`, `envelope_is_autoreply`, plus the activity's `lead_id`.

Note (verified Nov 1 2026): 100% of envelope emails are PII-masked (`user_XXXXXX@masked.com`). The mask is deterministic — 93% of distinct envelope emails match CRM contacts by exact masked email — so this table links email correspondence to contacts/leads without needing real addresses.

In [0]:
%run ../utils

In [0]:
# 2_2_04: envelope child table - one row per envelope email (all 6 directions), scalars carried
# Unpivoted on purpose: direction + email + name in one table is directly usable for
# identity resolution; six separate direction tables would need re-joining anyway.
# Option B CDC (parent-driven, engine utils): watermark on THIS child (bronze_updated_at
# carried down), changed parents only, LATEST version per activity_id, exact DELETE of
# affected activity_ids, then engine write_silver. First run / legacy schema = full rebuild.
from pyspark.sql.functions import col, explode_outer, lit, row_number
from pyspark.sql.window import Window

CHILD = 'lead_activites_processed_envelope'
try:
    watermark = get_silver_watermark(CHILD)
except Exception:
    watermark = None

src = spark.table('crm_ingestion.silver.lead_activites_processed').select(
    'id', 'lead_id', 'bronze_insert_date', 'bronze_updated_at', 'envelope')
if watermark is not None:
    wm_str = watermark.strftime('%Y-%m-%d %H:%M:%S.%f')
    print(f"CDC MODE: parents updated after {wm_str}")
    src = src.filter(col('bronze_updated_at') > watermark)
else:
    print("FULL LOAD: all activities, latest version per activity_id")

_version = Window.partitionBy('id').orderBy(col('bronze_updated_at').desc(), col('bronze_insert_date').desc())
df_in = src.withColumn('_v', row_number().over(_version)).filter(col('_v') == 1).drop('_v')

if watermark is None or len(df_in.take(1)) > 0:
    if watermark is not None:
        spark.sql(f"""
            DELETE FROM crm_ingestion.silver.{CHILD}
            WHERE activity_id IN (
                SELECT id FROM crm_ingestion.silver.lead_activites_processed
                WHERE bronze_updated_at > '{wm_str}'
            )""")
        _CDC_WATERMARK = watermark
    directions = ['from', 'to', 'cc', 'bcc', 'reply_to', 'sender']
    pieces = []
    for d in directions:
        pieces.append(
            df_in.select(
                col('id').alias('activity_id'),
                'lead_id', 'bronze_insert_date', 'bronze_updated_at',
                lit(d).alias('direction'),
                explode_outer(col('envelope.' + d)).alias('e'),
                col('envelope.subject').alias('envelope_subject'),
                col('envelope.message_id').alias('envelope_message_id'),
                col('envelope.is_autoreply').alias('envelope_is_autoreply')
            )
            .select(
                'activity_id', 'lead_id', 'bronze_insert_date', 'bronze_updated_at', 'direction',
                col('e.email').alias('email'),
                col('e.name').alias('name'),
                'envelope_subject', 'envelope_message_id', 'envelope_is_autoreply'
            )
        )
    df_envelope = pieces[0]
    for p in pieces[1:]:
        df_envelope = df_envelope.unionByName(p)
    df_envelope = df_envelope.dropna(subset=['email'])
    write_silver(df_envelope, CHILD)
    _CDC_WATERMARK = None
else:
    print("  No changed activities — nothing to do")